# Hate Speech Analysis

This is the final submission notebook for the hate-speech detection project. It presents the clean exploratory analysis, the preprocessing pipeline, and the final tuned `LinearSVC` model used for the project.

In [ ]:
from pathlib import Path
import pickle
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.svm import LinearSVC

from src.preprocess import LABEL_MAP, build_clean_dataframe, clean_text, load_raw_dataset

sns.set_theme(style="whitegrid")
pd.set_option("display.max_colwidth", 200)

In [ ]:
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

MODELS_DIR = PROJECT_ROOT / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
VECTORIZER_PATH = MODELS_DIR / "tfidf_vectorizer.pkl"
MODEL_PATH = MODELS_DIR / "hate_speech_svc.pkl"

df_raw = load_raw_dataset()
df = build_clean_dataframe(df_raw)

print("Dataset shape:", df.shape)
print("Columns:", list(df.columns))
df.head()

## 1. Clean EDA

The dataset is strongly imbalanced, so minority-class performance, especially the `hate_speech` class, matters more than raw accuracy.

In [ ]:
class_counts = df["class"].value_counts().sort_index()
class_df = pd.DataFrame({
    "class": class_counts.index,
    "label": [LABEL_MAP.get(i, str(i)) for i in class_counts.index],
    "count": class_counts.values,
    "percent": (class_counts.values / len(df) * 100).round(2),
})
class_df

In [ ]:
plt.figure(figsize=(8, 4))
ax = sns.barplot(data=class_df, x="label", y="count", hue="label", dodge=False, legend=False, palette="viridis")
ax.set_title("Class Distribution")
ax.set_xlabel("Class")
ax.set_ylabel("Count")
for i, count in enumerate(class_df["count"]):
    ax.text(i, count, str(count), ha="center", va="bottom", fontsize=9)
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

In [ ]:
df["char_len"] = df["tweet"].astype(str).str.len()
df["word_len"] = df["tweet"].astype(str).str.split().str.len()
df[["char_len", "word_len"]].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9, 0.99])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(df["char_len"], bins=50, kde=True, ax=axes[0], color="steelblue")
axes[0].set_title("Tweet Character Length Distribution")
axes[0].set_xlabel("Characters")
sns.histplot(df["word_len"], bins=40, kde=True, ax=axes[1], color="darkorange")
axes[1].set_title("Tweet Word Length Distribution")
axes[1].set_xlabel("Words")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.boxplot(data=df, x="class", y="char_len", ax=axes[0], palette="Set2")
axes[0].set_title("Character Length by Class")
sns.boxplot(data=df, x="class", y="word_len", ax=axes[1], palette="Set3")
axes[1].set_title("Word Length by Class")
plt.tight_layout()
plt.show()

In [ ]:
URL_RE = re.compile(r"https?://\\S+|www\\.\\S+", re.IGNORECASE)
MENTION_RE = re.compile(r"@\\w+")
HASHTAG_RE = re.compile(r"#\\w+")
RT_RE = re.compile(r"\\bRT\\b", re.IGNORECASE)

pattern_df = df.copy()
pattern_df["has_url"] = pattern_df["tweet"].astype(str).str.contains(URL_RE)
pattern_df["has_mention"] = pattern_df["tweet"].astype(str).str.contains(MENTION_RE)
pattern_df["has_hashtag"] = pattern_df["tweet"].astype(str).str.contains(HASHTAG_RE)
pattern_df["has_rt"] = pattern_df["tweet"].astype(str).str.contains(RT_RE)

pattern_summary = pd.DataFrame({
    "feature": ["has_url", "has_mention", "has_hashtag", "has_rt"],
    "count": [
        int(pattern_df["has_url"].sum()),
        int(pattern_df["has_mention"].sum()),
        int(pattern_df["has_hashtag"].sum()),
        int(pattern_df["has_rt"].sum()),
    ],
})
pattern_summary["percent"] = (pattern_summary["count"] / len(pattern_df) * 100).round(2)
pattern_summary

In [ ]:
plt.figure(figsize=(8, 4))
ax = sns.barplot(data=pattern_summary, x="feature", y="percent", hue="feature", dodge=False, legend=False, palette="magma")
ax.set_title("Tweet Structure Patterns in Raw Text")
ax.set_ylabel("Percent of tweets")
ax.set_xlabel("")
plt.tight_layout()
plt.show()

In [ ]:
def top_ngrams(corpus, ngram_range=(1, 1), top_k=12, min_df=2):
    vectorizer = CountVectorizer(stop_words="english", ngram_range=ngram_range, min_df=min_df)
    matrix = vectorizer.fit_transform(corpus)
    counts = np.asarray(matrix.sum(axis=0)).ravel()
    terms = np.array(vectorizer.get_feature_names_out())
    top_idx = counts.argsort()[::-1][:top_k]
    return pd.DataFrame({"term": terms[top_idx], "count": counts[top_idx]})

class_term_tables = []
for class_id in sorted(df["class"].unique()):
    subset = df.loc[df["class"] == class_id, "clean_tweet"]
    top_terms = top_ngrams(subset, ngram_range=(1, 1), top_k=10, min_df=2)
    top_terms.insert(0, "label", LABEL_MAP[class_id])
    class_term_tables.append(top_terms)

pd.concat(class_term_tables, ignore_index=True)

## 2. Preprocessing Pipeline

The final text pipeline removes URLs, mentions, the `RT` token, punctuation and non-letter symbols, then lowercases and compresses repeated whitespace before TF-IDF vectorization.

In [ ]:
sample_rows = df[["tweet", "clean_tweet"]].head(5).copy()
sample_rows

In [ ]:
vectorizer = TfidfVectorizer(
    lowercase=False,
    ngram_range=(1, 2),
    min_df=2,
    max_features=20000,
)
vectorizer

## 3. Winning Model: Tuned LinearSVC

Because the dataset is imbalanced, the final model uses `class_weight='balanced'`. The key result to watch is the `Class 0` F1-score because class `0` corresponds to `hate_speech`.

In [ ]:
X_train_text, X_test_text, y_train, y_test = train_test_split(
    df["clean_tweet"],
    df["class"],
    test_size=0.2,
    random_state=42,
    stratify=df["class"],
)

X_train = vectorizer.fit_transform(X_train_text)
X_test = vectorizer.transform(X_test_text)

svc = LinearSVC(
    C=0.5,
    class_weight="balanced",
    loss="hinge",
    max_iter=15000,
    random_state=42,
)
svc.fit(X_train, y_train)
y_pred = svc.predict(X_test)

print("Train matrix:", X_train.shape)
print("Test matrix:", X_test.shape)

In [ ]:
report_dict = classification_report(y_test, y_pred, output_dict=True, zero_division=0)
metrics = {
    "accuracy": accuracy_score(y_test, y_pred),
    "precision_macro": precision_score(y_test, y_pred, average="macro", zero_division=0),
    "recall_macro": recall_score(y_test, y_pred, average="macro", zero_division=0),
    "f1_macro": f1_score(y_test, y_pred, average="macro", zero_division=0),
    "f1_weighted": f1_score(y_test, y_pred, average="weighted", zero_division=0),
    "class0_precision": report_dict["0"]["precision"],
    "class0_recall": report_dict["0"]["recall"],
    "class0_f1": report_dict["0"]["f1-score"],
}
pd.DataFrame([metrics]).round(4)

In [ ]:
print(classification_report(y_test, y_pred, digits=4, zero_division=0))

In [ ]:
class0_summary = pd.DataFrame([
    {
        "focus_metric": "Class 0 (hate_speech) F1-score",
        "value": report_dict["0"]["f1-score"],
    },
    {
        "focus_metric": "Class 0 (hate_speech) Recall",
        "value": report_dict["0"]["recall"],
    },
    {
        "focus_metric": "Macro F1",
        "value": f1_score(y_test, y_pred, average="macro", zero_division=0),
    },
])
class0_summary.round(4)

In [ ]:
labels = sorted(df["class"].unique())
cm = confusion_matrix(y_test, y_pred, labels=labels)
cm_df = pd.DataFrame(cm, index=[f"true_{i}" for i in labels], columns=[f"pred_{i}" for i in labels])
cm_df

In [ ]:
plt.figure(figsize=(5, 4))
sns.heatmap(cm_df, annot=True, fmt="d", cmap="Blues")
plt.title("Confusion Matrix - Tuned LinearSVC")
plt.tight_layout()
plt.show()

In [ ]:
with VECTORIZER_PATH.open("wb") as vectorizer_file:
    pickle.dump(vectorizer, vectorizer_file)
with MODEL_PATH.open("wb") as model_file:
    pickle.dump(svc, model_file)
print(f"Saved vectorizer to {VECTORIZER_PATH}")
print(f"Saved model to {MODEL_PATH}")

## 4. Ablation Summary

The classical `LinearSVC` was selected as the final model because it gave the best trade-off between macro F1 and minority-class detection. The TF-IDF representation is very strong for short, noisy tweets, and the linear margin-based model handled the sparse feature space efficiently.

The deep learning attempts were useful experiments, but they did not beat the final SVC on the core evaluation goal. The MLP needed dimensionality reduction to remain stable on sparse TF-IDF inputs, and the embedding-based CNN required more tuning and data efficiency than this project setting allowed. In practice, the tuned SVC remained the most reliable model for improving the `hate_speech` class F1-score while keeping the training pipeline simple and reproducible.